In [ ]:
import folium
from folium.plugins import MarkerCluster
import json
import pandas as pd
import html as html_module
import re
import urllib.request
from math import radians, cos, sin, sqrt
from pathlib import Path
from IPython.display import display, HTML

# =========================================================
# CONFIG
# =========================================================
BASE_DIR    = Path(".")
TEXTS_DIR   = BASE_DIR / "data_texts"
FR_PATH     = TEXTS_DIR / "20000lieues_fr.txt"
EN_PATH     = TEXTS_DIR / "20000lieues_an.txt"
JSON_PATH   = BASE_DIR / "Points-escales-chapitres.json"
OUTPUT_HTML = BASE_DIR / "nautilus_map_v16.html"

# ── TEST MODE : 5 premières escales seulement ──
TEST_MODE      = True
TEST_NB_ESCALES = 5

for p in [TEXTS_DIR, FR_PATH, EN_PATH, JSON_PATH]:
    assert p.exists(), f"❌ Introuvable : {p.resolve()}"
    print(f"✅ {p.resolve()}")

fr_text_full = FR_PATH.read_text(encoding="utf-8", errors="ignore")
en_text_full = EN_PATH.read_text(encoding="utf-8", errors="ignore")
print(f"📖 FR : {len(fr_text_full):,} caractères")
print(f"📖 EN : {len(en_text_full):,} caractères")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

if TEST_MODE:
    data = data[:TEST_NB_ESCALES]
    print(f"🧪 MODE TEST — {TEST_NB_ESCALES} escales seulement")

print(f"🗺️  {len(data)} escales chargées")

In [ ]:
# =========================================================
# UTILITAIRES
# =========================================================
def clean_text(txt):
    if txt is None: return ""
    txt = str(txt)
    txt = txt.replace("\u00a0", " ").replace("\ufeff", "")
    txt = re.sub(r"\r\n?", "\n", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    return txt.strip()

def get_lang(value, lang):
    if isinstance(value, dict):
        v = value.get(lang) or value.get("fr") or ""
        return clean_text(str(v)) if v else ""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return clean_text(str(value))

def distance_from_center(lat):
    R = 6371e3
    lat_rad = radians(lat)
    return int(sqrt((R * cos(lat_rad))**2 + (R * sin(lat_rad))**2))

# =========================================================
# EXTRACTION DU CHAPITRE DEPUIS LE TEXTE
# =========================================================
ROMAINS_FR = {
    1: "PREMIER", 2: "II", 3: "III", 4: "IV", 5: "V",
    6: "VI", 7: "VII", 8: "VIII", 9: "IX", 10: "X",
    11: "XI", 12: "XII", 13: "XIII", 14: "XIV", 15: "XV",
    16: "XVI", 17: "XVII", 18: "XVIII", 19: "XIX", 20: "XX",
    21: "XXI", 22: "XXII", 23: "XXIII", 24: "XXIV"
}
ROMAINS_EN = {
    1: "I", 2: "II", 3: "III", 4: "IV", 5: "V",
    6: "VI", 7: "VII", 8: "VIII", 9: "IX", 10: "X",
    11: "XI", 12: "XII", 13: "XIII", 14: "XIV", 15: "XV",
    16: "XVI", 17: "XVII", 18: "XVIII", 19: "XIX", 20: "XX",
    21: "XXI", 22: "XXII", 23: "XXIII", 24: "XXIV"
}

def extraire_chapitre_fr(texte, partie, numero):
    romain = ROMAINS_FR.get(numero, str(numero))
    marqueur = f"CHAPITRE {romain}"
    split_partie = texte.split("FIN DE LA PREMIÈRE PARTIE")
    zone = split_partie[0] if partie == 1 else (split_partie[1] if len(split_partie) > 1 else texte)
    idx_debut = zone.find(marqueur)
    if idx_debut == -1: return None
    idx_suite = zone.find("CHAPITRE", idx_debut + len(marqueur))
    return zone[idx_debut:idx_suite].strip() if idx_suite != -1 else zone[idx_debut:].strip()

def extraire_chapitre_en(texte, partie, numero):
    romain = ROMAINS_EN.get(numero, str(numero))
    marqueur = f"CHAPTER {romain}"
    split_partie = texte.split("PART TWO")
    zone = split_partie[0] if partie == 1 else (split_partie[1] if len(split_partie) > 1 else texte)
    idx_debut = zone.find(marqueur)
    if idx_debut == -1: return None
    idx_suite = zone.find("CHAPTER", idx_debut + len(marqueur))
    return zone[idx_debut:idx_suite].strip() if idx_suite != -1 else zone[idx_debut:].strip()

print("✅ Utilitaires chargés")

In [ ]:
# =========================================================
# OLLAMA — GÉNÉRATION EXTRAIT FICHE 1
# =========================================================
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

def appeler_ollama(prompt, num_predict=2500):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2, "num_predict": num_predict}
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            result = json.loads(resp.read().decode("utf-8"))
            return result.get("response", "").strip()
    except Exception as e:
        return None

def prompt_extrait_fr(nom_escale, chapitre_texte):
    return f"""Tu es un assistant littéraire spécialisé dans Jules Verne.

Voici le texte du chapitre correspondant à l'escale \"{nom_escale}\" dans \"Vingt mille lieues sous les mers\".

TEXTE :
{chapitre_texte[:6000]}

INSTRUCTIONS STRICTES :
- Sélectionne entre 4 et 6 paragraphes riches et représentatifs de ce chapitre.
- Copie les paragraphes EXACTEMENT tels qu'ils apparaissent dans le texte, sans les modifier ni les résumer.
- Ne coupe jamais une phrase en plein milieu.
- Le texte doit être suffisamment long pour donner une vraie immersion dans le chapitre.
- Si le nom de l'escale ({nom_escale}) apparaît dans le texte, inclus le paragraphe qui le contient.
- Sépare chaque paragraphe par une ligne vide.
- Ne rajoute AUCUN commentaire, titre, introduction ou conclusion. Uniquement les paragraphes du roman.
"""

def prompt_extrait_en(nom_escale, chapitre_texte):
    return f"""You are a literary assistant specialized in Jules Verne.

Here is the chapter text corresponding to the stop \"{nom_escale}\" in \"Twenty Thousand Leagues Under the Sea\".

TEXT:
{chapitre_texte[:6000]}

STRICT INSTRUCTIONS:
- Select between 4 and 6 rich and representative paragraphs from this chapter.
- Copy the paragraphs EXACTLY as they appear in the text, without modifying or summarizing them.
- Never cut a sentence in the middle.
- The text must be long enough to provide true immersion in the chapter.
- If the stop name ({nom_escale}) appears in the text, include the paragraph containing it.
- Separate each paragraph with a blank line.
- Do NOT add ANY comment, title, introduction or conclusion. Only the novel's paragraphs.
"""

def vers_html_extrait(texte_brut, nom_escale, lang):
    """Convertit le texte brut Ollama en HTML HUD."""
    if not texte_brut:
        msg = "Extrait non disponible." if lang == "fr" else "Excerpt unavailable."
        return f"<p><em>{msg}</em></p>"
    paragraphes = [p.strip() for p in texte_brut.split("\n\n") if p.strip()]
    # Filtre les lignes trop courtes (titres de chapitres)
    paragraphes = [p for p in paragraphes if len(p) > 80]
    html = ""
    for p in paragraphes:
        html += f"<p style='margin-bottom:1.4rem; line-height:1.85; font-size:1rem;'>{p}</p>\n"
    return html

# ── Génération pour chaque escale ──
import time
extraits_content = {}
MSG_INTER_FR = "<p><em>Point de navigation intermédiaire — aucun chapitre spécifique dans le roman.</em></p>"
MSG_INTER_EN = "<p><em>Intermediate navigation point — no specific chapter in the novel.</em></p>"

for i, escale in enumerate(data):
    nom_fr = get_lang(escale.get("Escales du Nautilus"), "fr")
    nom_en = get_lang(escale.get("Escales du Nautilus"), "en")
    ch_info = escale.get("chapitre")

    print(f"[{i+1}/{len(data)}] 🧭 {nom_fr}")

    if not ch_info:
        extraits_content[str(i)] = {"fr": MSG_INTER_FR, "en": MSG_INTER_EN}
        print("  ⏭️  Point intermédiaire")
        continue

    partie = ch_info["partie"]
    numero = ch_info["chapitre"]

    ch_fr = extraire_chapitre_fr(fr_text_full, partie, numero)
    ch_en = extraire_chapitre_en(en_text_full, partie, numero)

    print(f"  📄 FR : {len(ch_fr) if ch_fr else 0} car. | EN : {len(ch_en) if ch_en else 0} car.")

    print("  🤖 Génération FR...")
    raw_fr = appeler_ollama(prompt_extrait_fr(nom_fr, ch_fr)) if ch_fr else None
    time.sleep(1)

    print("  🤖 Génération EN...")
    raw_en = appeler_ollama(prompt_extrait_en(nom_en, ch_en)) if ch_en else None
    time.sleep(1)

    extraits_content[str(i)] = {
        "fr": vers_html_extrait(raw_fr, nom_fr, "fr"),
        "en": vers_html_extrait(raw_en, nom_en, "en")
    }
    print(f"  ✅ OK")

print(f"\n✅ {len(extraits_content)} extraits générés")

In [ ]:
# =========================================================
# DATAFRAME
# =========================================================
df = pd.DataFrame(data)
df["Latitude_Décimal"]  = pd.to_numeric(df["Latitude_Décimal"],  errors="coerce")
df["Longitude_Décimal"] = pd.to_numeric(df["Longitude_Décimal"], errors="coerce")
df = df.dropna(subset=["Latitude_Décimal", "Longitude_Décimal"]).reset_index(drop=True)
df["DistanceCentreTerre"] = df["Latitude_Décimal"].apply(distance_from_center)
print(f"✅ {len(df)} escales avec coordonnées valides")

In [ ]:
# =========================================================
# POPUP HUD — style noir/turquoise, sans Bootstrap
# =========================================================
def make_popup(row, i):
    stop_fr  = html_module.escape(get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}")
    date_fr  = html_module.escape(get_lang(row.get("Date"),      "fr"))
    ocean_fr = html_module.escape(get_lang(row.get("Océan/Mer"), "fr"))
    event_fr = html_module.escape(get_lang(row.get("Event"),     "fr"))
    stop_en  = html_module.escape(get_lang(row.get("Escales du Nautilus"), "en") or f"Stop {i+1}")
    date_en  = html_module.escape(get_lang(row.get("Date"),      "en"))
    ocean_en = html_module.escape(get_lang(row.get("Océan/Mer"), "en"))
    event_en = html_module.escape(get_lang(row.get("Event"),     "en"))
    lat  = row["Latitude_Décimal"]
    lon  = row["Longitude_Décimal"]
    dist = f"{row['DistanceCentreTerre']:,}"

    return f"""
    <div class="hud-popup">

      <!-- FRANÇAIS -->
      <div class="hud-col">
        <div class="hud-lang-label">Français 🇫🇷</div>
        <div class="hud-body">
          <div class="hud-row">🗺️ <span class="hud-key">Escale :</span> <span class="hud-val">{stop_fr}</span></div>
          <div class="hud-row">📅 <span class="hud-key">Date :</span> <span class="hud-val">{date_fr}</span></div>
          <div class="hud-row">🌊 <span class="hud-key">Océan/Mer :</span> <span class="hud-val">{ocean_fr}</span></div>
          <div class="hud-row">📜 <span class="hud-key">Événement :</span> <span class="hud-val">{event_fr}</span></div>
          <div class="hud-row">↕️ <span class="hud-key">Latitude :</span> <span class="hud-val">{lat}</span></div>
          <div class="hud-row">↔️ <span class="hud-key">Longitude :</span> <span class="hud-val">{lon}</span></div>
          <div class="hud-row">🌍 <span class="hud-key">Distance centre Terre :</span> <span class="hud-val">{dist} m</span></div>
        </div>
        <button class="hud-btn" data-stop="{i}" data-lang="fr" data-action="book_excerpt">
          📖 Journal de bord
        </button>
      </div>

      <div class="hud-divider"></div>

      <!-- ENGLISH -->
      <div class="hud-col">
        <div class="hud-lang-label">English 🇬🇧</div>
        <div class="hud-body">
          <div class="hud-row">🗺️ <span class="hud-key">Stop :</span> <span class="hud-val">{stop_en}</span></div>
          <div class="hud-row">📅 <span class="hud-key">Date :</span> <span class="hud-val">{date_en}</span></div>
          <div class="hud-row">🌊 <span class="hud-key">Ocean/Sea :</span> <span class="hud-val">{ocean_en}</span></div>
          <div class="hud-row">📜 <span class="hud-key">Event :</span> <span class="hud-val">{event_en}</span></div>
          <div class="hud-row">↕️ <span class="hud-key">Latitude :</span> <span class="hud-val">{lat}</span></div>
          <div class="hud-row">↔️ <span class="hud-key">Longitude :</span> <span class="hud-val">{lon}</span></div>
          <div class="hud-row">🌍 <span class="hud-key">Distance from Earth's center :</span> <span class="hud-val">{dist} m</span></div>
        </div>
        <button class="hud-btn" data-stop="{i}" data-lang="en" data-action="book_excerpt">
          📖 Living Logbook
        </button>
      </div>

    </div>"""

print("✅ make_popup HUD prêt")

In [ ]:
# =========================================================
# CARTE FOLIUM — tiles sombres
# =========================================================
m = folium.Map(
    location=[20, -30],
    zoom_start=2,
    tiles="CartoDB dark_matter"  # fond sombre cohérent avec le HUD
)
marker_cluster = MarkerCluster().add_to(m)
coords = []

for i, row in df.iterrows():
    lat = row["Latitude_Décimal"]
    lon = row["Longitude_Décimal"]
    coords.append([lat, lon])
    tooltip = html_module.escape(get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}")

    if i == 0:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=660),
            tooltip=f"⚓ Départ — {tooltip}",
            icon=folium.Icon(color="green", icon="anchor", prefix="fa")
        ).add_to(m)
    elif i == len(df) - 1:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=660),
            tooltip=f"🏁 Arrivée — {tooltip}",
            icon=folium.Icon(color="red", icon="flag", prefix="fa")
        ).add_to(m)
    else:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=660),
            tooltip=tooltip,
            icon=folium.Icon(color="cadetblue", icon="info-sign")
        ).add_to(marker_cluster)

if len(coords) > 1:
    folium.PolyLine(coords, color="#27c39f", weight=2, opacity=0.7).add_to(m)

print(f"✅ {len(coords)} marqueurs — trajet turquoise")

In [1]:
# =========================================================
# INJECTION HTML — CSS HUD + Modal + JS
# =========================================================
custom_html = f"""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Michroma&display=swap" rel="stylesheet">

<style>
/* ── Popup leaflet ── */
.leaflet-popup-content-wrapper {{
  background: rgba(5, 15, 25, 0.96) !important;
  border: 1px solid rgba(39,195,159,0.30) !important;
  border-radius: 18px !important;
  box-shadow: 0 8px 32px rgba(0,0,0,0.55), inset 0 0 24px rgba(39,195,159,0.04) !important;
  backdrop-filter: blur(12px);
}}
.leaflet-popup-tip {{
  background: rgba(5,15,25,0.96) !important;
}}
.leaflet-popup-content {{ margin: 0 !important; padding: 0 !important; }}

/* ── Layout popup ── */
.hud-popup {{
  display: flex;
  gap: 0;
  min-width: 560px;
  font-family: 'Raleway', Arial, sans-serif;
}}
.hud-col {{
  flex: 1;
  padding: 18px 20px 16px;
  display: flex;
  flex-direction: column;
  gap: 0;
}}
.hud-divider {{
  width: 1px;
  background: rgba(39,195,159,0.20);
  margin: 14px 0;
}}
.hud-lang-label {{
  font-family: 'Michroma', sans-serif;
  font-size: 0.72rem;
  letter-spacing: 0.18em;
  text-transform: uppercase;
  color: #a8ffff;
  margin-bottom: 12px;
}}
.hud-body {{
  display: flex;
  flex-direction: column;
  gap: 6px;
  flex: 1;
}}
.hud-row {{
  font-size: 0.82rem;
  color: #c8dde8;
  line-height: 1.55;
}}
.hud-key {{
  color: #27c39f;
  font-weight: 600;
}}
.hud-val {{
  color: #e0f4ff;
}}

/* ── Bouton HUD ── */
.hud-btn {{
  margin-top: 14px;
  padding: 7px 14px;
  background: transparent;
  border: 1px solid rgba(39,195,159,0.5);
  border-radius: 8px;
  color: #27c39f;
  font-family: 'Michroma', sans-serif;
  font-size: 0.72rem;
  letter-spacing: 0.10em;
  cursor: pointer;
  transition: all 0.2s ease;
  position: relative;
  overflow: hidden;
}}
.hud-btn:hover {{
  background: rgba(39,195,159,0.12);
  border-color: #27c39f;
  box-shadow: 0 0 12px rgba(39,195,159,0.25);
}}
/* shimmer au chargement */
.hud-btn.loading {{
  pointer-events: none;
  color: transparent;
}}
.hud-btn.loading::after {{
  content: '';
  position: absolute;
  inset: 0;
  background: linear-gradient(90deg,
    rgba(39,195,159,0) 0%,
    rgba(39,195,159,0.35) 50%,
    rgba(39,195,159,0) 100%);
  animation: shimmer 1.2s ease-in-out infinite;
}}
@keyframes shimmer {{
  0%   {{ transform: translateX(-100%); }}
  100% {{ transform: translateX(100%); }}
}}

/* ── Modal HUD ── */
#hudModal {{
  display: none;
  position: fixed;
  inset: 0;
  z-index: 9999;
  align-items: center;
  justify-content: center;
  background: rgba(0,0,0,0.60);
  backdrop-filter: blur(4px);
}}
#hudModal.open {{ display: flex; }}
#hudModalBox {{
  position: relative;
  width: min(860px, 92vw);
  max-height: 85vh;
  background: linear-gradient(160deg, rgba(5,18,28,0.98), rgba(3,10,18,0.96));
  border: 1px solid rgba(168,255,255,0.22);
  border-radius: 22px;
  box-shadow: 0 24px 60px rgba(0,0,0,0.5), inset 0 0 30px rgba(39,195,159,0.04);
  display: flex;
  flex-direction: column;
  overflow: hidden;
}}
#hudModalHeader {{
  padding: 16px 22px 14px;
  border-bottom: 1px solid rgba(39,195,159,0.18);
  display: flex;
  align-items: center;
  justify-content: space-between;
}}
#hudModalTitle {{
  font-family: 'Michroma', sans-serif;
  font-size: 0.9rem;
  letter-spacing: 0.14em;
  color: #27c39f;
  text-transform: uppercase;
}}
#hudModalClose {{
  background: none;
  border: none;
  color: #a8ffff;
  font-size: 1.3rem;
  cursor: pointer;
  opacity: 0.7;
  transition: opacity 0.2s;
}}
#hudModalClose:hover {{ opacity: 1; }}
#hudModalBody {{
  padding: 24px 28px;
  overflow-y: auto;
  color: #d5edf5;
  font-family: 'Raleway', Arial, sans-serif;
  font-size: 0.97rem;
  line-height: 1.85;
  scrollbar-width: thin;
  scrollbar-color: #27c39f rgba(255,255,255,0.06);
}}
#hudModalBody::-webkit-scrollbar {{ width: 8px; }}
#hudModalBody::-webkit-scrollbar-track {{ background: rgba(255,255,255,0.04); border-radius: 999px; }}
#hudModalBody::-webkit-scrollbar-thumb {{ background: linear-gradient(#27c39f,#195480); border-radius: 999px; }}
#hudModalBody p {{ margin-bottom: 1.3rem; }}
</style>

<!-- Modal HUD -->
<div id="hudModal">
  <div id="hudModalBox">
    <div id="hudModalHeader">
      <span id="hudModalTitle">Journal de bord</span>
      <button id="hudModalClose" onclick="closeHudModal()">✕</button>
    </div>
    <div id="hudModalBody"></div>
  </div>
</div>

<script>
const EXTRAITS = {json.dumps(extraits_content, ensure_ascii=False)};

const MODAL_TITLES = {{
  fr: {{ book_excerpt: "📖 Journal de bord vivant" }},
  en: {{ book_excerpt: "📖 Living Logbook" }}
}};

function openHudModal(title, content) {{
  document.getElementById("hudModalTitle").innerHTML = title;
  document.getElementById("hudModalBody").innerHTML  = content;
  document.getElementById("hudModal").classList.add("open");
}}

function closeHudModal() {{
  document.getElementById("hudModal").classList.remove("open");
}}

// Ferme en cliquant hors de la modal
document.getElementById("hudModal").addEventListener("click", function(e) {{
  if (e.target === this) closeHudModal();
}});

// Délégation de clic sur les boutons HUD dans les popups
document.addEventListener("click", function(e) {{
  const btn = e.target.closest(".hud-btn");
  if (!btn) return;

  const stop   = btn.getAttribute("data-stop");
  const lang   = btn.getAttribute("data-lang") || "fr";
  const action = btn.getAttribute("data-action");

  // Shimmer pendant le "chargement"
  btn.classList.add("loading");

  setTimeout(function() {{
    btn.classList.remove("loading");

    const fallback = lang === "fr"
      ? "<p><em>Contenu disponible bientôt.</em></p>"
      : "<p><em>Content coming soon.</em></p>";

    let content = fallback;
    if (action === "book_excerpt" && EXTRAITS[stop]) {{
      content = EXTRAITS[stop][lang] || fallback;
    }}

    const title = (MODAL_TITLES[lang] && MODAL_TITLES[lang][action])
      ? MODAL_TITLES[lang][action]
      : "Journal de bord";

    openHudModal(title, content);
  }}, 600);  // délai shimmer 600ms
}});
</script>
"""

m.get_root().html.add_child(folium.Element(custom_html))
m.save(str(OUTPUT_HTML))
print(f"✅ Fichier généré : {OUTPUT_HTML.resolve()}")
display(HTML(f'<a href="{OUTPUT_HTML.name}" target="_blank">🔗 Ouvrir la carte HTML</a>')) 

NameError: name 'json' is not defined